# UbiFusionNet v1 — same-data publication experiment

This notebook trains the proposed architecture on the exact released MMUbiPred training samples. Training and model selection do not access the locked independent test set. Use a GPU runtime.

## Repository access

If this GitHub repository is private, create a fine-grained read-only GitHub token, add it to Colab under **Secrets** with the name `GITHUB_TOKEN`, and enable notebook access. The token is supplied through a temporary Git askpass helper; it is not placed in the clone URL or saved remote. A public repository needs no secret.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. For a private repository, add a read-only '
        'GITHUB_TOKEN in the Colab Secrets panel and enable notebook access.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt

In [ ]:
import platform
import sys
import numpy as np
import sklearn
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('NumPy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Audit immutable benchmark artifacts

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/audit_dataset.py',
    '--data-dir', 'replication/MMUbiPred',
    '--output', 'experiment/outputs/dataset_audit.json',
])

## Select on development data, then refit on the complete released training set

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/train.py',
    '--config', 'experiment/configs/ubifusion_v1.json',
])

In [ ]:
import json

RUN_DIR = REPO_ROOT / 'experiment/outputs/ubifusion_v1_seed42'
validation_metrics = json.loads((RUN_DIR / 'validation_metrics.json').read_text())
validation_metrics

## Locked independent-test evaluation

Leave `RUN_LOCKED_TEST=False` during architecture development. Set it to `True` only after this configuration has been accepted from validation results.

In [ ]:
RUN_LOCKED_TEST = False

if RUN_LOCKED_TEST:
    run_live([
        sys.executable, '-u', 'experiment/scripts/evaluate.py',
        '--run-dir', str(RUN_DIR),
        '--data-dir', 'replication/MMUbiPred',
        '--allow-locked-test',
    ])
    locked_metrics = json.loads((RUN_DIR / 'locked_test_metrics.json').read_text())
    display(locked_metrics)
else:
    print('Locked test was not accessed.')

## Preserve the run in Google Drive

In [ ]:
import shutil
from datetime import date

DRIVE_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments') / f'{date.today().isoformat()}-ubifusion-v1-seed42'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
for artifact in RUN_DIR.iterdir():
    destination = DRIVE_RUN_DIR / artifact.name
    if artifact.is_dir():
        shutil.copytree(artifact, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(artifact, destination)
print('Saved run artifacts to:', DRIVE_RUN_DIR)